# Student-t 重尾误差下的 DML 稳健性模拟（含正态基准，$p=5$）

本 notebook **不包含 contamination（混合污染）机制**，研究结果方程误差 $U$ 服从 Student-t 分布时，普通 Mean DML 与稳健 Median DML 的有限样本表现，并加入 $N(0,1)$ 作为同方差 baseline。

部分线性模型为
$$Y=D\theta_0+g_0(X)+U,\qquad D=m_0(X)+V.$$

为公平比较不同自由度，使用 $U=T_\nu\sqrt{(\nu-2)/\nu}$（$\nu>2$），使所有场景均满足 $\operatorname{Var}(U)=1$。因此自由度变化主要反映尾部厚度，而不是误差方差变化。Student-t 是对称分布，所以 $U$ 的均值与中位数均为 0。

In [ ]:
# ============================================================
# 调参区：正式实验通常只需要修改这里
# ============================================================
P = 5                              # 本 notebook 固定 p=5
N = 1000                           # 正式实验每次重复的样本量
THETA_0 = 1.0                      # 真实处理效应
T_DF_GRID = [2.1, 2.3, 2.6, 3, 5, 10, 20]  # Student-t 自由度；越小表示尾部越厚，且必须 > 2
NORMAL_BASELINE = "N(0,1)"             # 与标准化 Student-t 同为均值 0、方差 1
ERROR_GRID = [*T_DF_GRID, NORMAL_BASELINE] # 模拟顺序：重尾逐渐变轻，最后为正态基准
DGP_NAMES = ["DGP1_linear", "DGP2_nonlinear"]

N_FOLDS = 5                        # cross-fitting 折数
N_REPLICATIONS = 1000               # 正式 Monte Carlo 重复次数
BASE_SEED = 20260826               # 总随机种子，保证结果可复现
N_JOBS = -1                        # -1 使用全部 CPU；1 为串行

# 先用快速模式跑通全部场景；正式模拟时改为 False
QUICK_TEST = False
QUICK_N = 300                      # 快速模式样本量，仅用于检查代码
QUICK_REPLICATIONS = 2

# 两类 nuisance learner 共用超参数，保证比较口径一致
BOOSTING_PARAMS = {
    "learning_rate": 0.05,
    "max_iter": 100,
    "max_leaf_nodes": 15,
    "min_samples_leaf": 20,
    "l2_regularization": 1.0,
}

# 默认不写出 CSV，避免桌面产生额外运行文件
SAVE_RESULTS = True
RESULTS_FILE = "dml_p5_student_t_raw.csv"
SUMMARY_FILE = "dml_p5_student_t_summary.csv"

In [ ]:
# 依赖均来自常见 Python 科学计算环境；如缺包，请在终端中安装后重新启动内核
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed
from scipy.stats import norm
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold

warnings.filterwarnings("ignore", category=RuntimeWarning)
sns.set_theme(style="whitegrid", context="notebook")

assert P == 5, "本 notebook 专门用于 p=5；如改维度，请同步检查 DGP 公式。"
assert all(df > 2 for df in T_DF_GRID), "方差归一化要求所有 Student-t 自由度都大于 2。"
assert len(ERROR_GRID) == len(set(ERROR_GRID)), "误差场景不能重复。"
print("依赖与参数检查完成。")

## 1. 两种 DGP、Student-t 重尾误差与正态基准

沿用原项目的协变量相关结构与两种 nuisance functions。误差场景包括方差标准化的 Student-t 网格以及 $N(0,1)$ baseline；$X$、$D$、$V$ 和真实参数均保持不变。

In [ ]:
def ar1_covariance(p: int, rho: float = 0.5) -> np.ndarray:
    """构造 Sigma[j,k] = rho**|j-k| 的 AR(1) 型协方差矩阵。"""
    index = np.arange(p)
    return rho ** np.abs(index[:, None] - index[None, :])


def nuisance_functions(X: np.ndarray, dgp_name: str):
    """返回指定 DGP 的 m0(X) 与 g0(X)。"""
    x1, x2, x3, x4, x5 = X.T

    if dgp_name == "DGP1_linear":
        m0 = 0.80 * x1 - 0.50 * x2 + 0.30 * x3 + 0.20 * x4 - 0.10 * x5
        g0 = 1.00 * x1 + 0.50 * x2 - 0.50 * x3 + 0.25 * x4 + 0.10 * x5
    elif dgp_name == "DGP2_nonlinear":
        # x2**2 - 1 做中心化，因为标准正态变量满足 E[X^2]=1
        m0 = (
            0.60 * np.sin(x1)
            + 0.35 * (x2**2 - 1.0)
            + 0.25 * x3 * x4
            - 0.20 * np.tanh(x5)
        )
        g0 = (
            1.20 * np.sin(x1)
            + 0.70 * (x2**2 - 1.0)
            + 0.50 * x3 * x4
            + 0.30 * np.tanh(x5)
        )
    else:
        raise ValueError(f"未知 DGP: {dgp_name}")
    return m0, g0


def standardized_student_t(rng: np.random.Generator, df: float, size: int) -> np.ndarray:
    """生成均值 0、方差 1 的 Student-t 误差；要求 df > 2。"""
    if df <= 2:
        raise ValueError("Student-t 自由度必须大于 2，才能标准化为单位方差。")
    return rng.standard_t(df, size=size) * np.sqrt((df - 2.0) / df)


def generate_data(n: int, error_case, dgp_name: str, seed: int, theta0: float = THETA_0):
    """生成一个 Student-t 或 N(0,1) 结果误差下的部分线性模型样本。"""
    rng = np.random.default_rng(seed)
    sigma_x = ar1_covariance(P, rho=0.5)
    X = rng.multivariate_normal(np.zeros(P), sigma_x, size=n)
    m0, g0 = nuisance_functions(X, dgp_name)

    V = rng.normal(0.0, 1.0, size=n)
    if error_case == NORMAL_BASELINE:
        U = rng.normal(0.0, 1.0, size=n)
    else:
        U = standardized_student_t(rng, df=float(error_case), size=n)
    D = m0 + V
    Y = theta0 * D + g0 + U
    return X, D, Y, U


# 单次抽样检查：重尾样本中偶尔会出现较大极值；小样本方差不必恰好等于 1
X_demo, D_demo, Y_demo, U_demo = generate_data(
    n=N, error_case=T_DF_GRID[0], dgp_name=DGP_NAMES[0], seed=BASE_SEED
)
print(
    f"X shape: {X_demo.shape}; df={T_DF_GRID[0]}; "
    f"U mean={U_demo.mean():.3f}; U variance={U_demo.var(ddof=1):.3f}; "
    f"max|U|={np.max(np.abs(U_demo)):.2f}"
)

## 2. Cross-fitted DML 估计量

- `Mean_DML`：第一阶段拟合条件均值，第二阶段在交叉拟合残差上做过原点 OLS。
- `Median_DML`：第一阶段拟合条件中位数，第二阶段使用 LAD 斜率。

每个观测的 nuisance 预测都来自没有见过该观测的训练折，避免用训练内预测替代 cross-fitting。

In [ ]:
def make_nuisance_model(kind: str, seed: int):
    """根据估计量类型创建条件均值或条件中位数学习器。"""
    common = dict(BOOSTING_PARAMS)
    common["random_state"] = seed
    if kind == "mean":
        return HistGradientBoostingRegressor(loss="squared_error", **common)
    if kind == "median":
        return HistGradientBoostingRegressor(loss="quantile", quantile=0.5, **common)
    raise ValueError("kind 必须是 'mean' 或 'median'。")


def cross_fitted_residuals(X, D, Y, kind: str, n_folds: int, seed: int):
    """K-fold cross-fitting：返回处理与结果的 out-of-fold 残差。"""
    d_hat = np.empty_like(D, dtype=float)
    y_hat = np.empty_like(Y, dtype=float)
    splitter = KFold(n_splits=n_folds, shuffle=True, random_state=seed)

    for fold, (train_idx, test_idx) in enumerate(splitter.split(X)):
        model_d = make_nuisance_model(kind, seed + 100 * fold + 1)
        model_y = make_nuisance_model(kind, seed + 100 * fold + 2)
        model_d.fit(X[train_idx], D[train_idx])
        model_y.fit(X[train_idx], Y[train_idx])
        d_hat[test_idx] = model_d.predict(X[test_idx])
        y_hat[test_idx] = model_y.predict(X[test_idx])

    return D - d_hat, Y - y_hat


def weighted_median(values: np.ndarray, weights: np.ndarray) -> float:
    """计算加权中位数；用于求 LAD 过原点斜率。"""
    order = np.argsort(values)
    values_sorted = values[order]
    weights_sorted = weights[order]
    cutoff = 0.5 * weights_sorted.sum()
    index = np.searchsorted(np.cumsum(weights_sorted), cutoff, side="left")
    return float(values_sorted[min(index, len(values_sorted) - 1)])


def fit_mean_dml(d_res: np.ndarray, y_res: np.ndarray):
    """第二阶段残差 OLS，并由正交得分的经验影响函数计算标准误。"""
    jacobian = np.mean(d_res**2)
    if jacobian <= 1e-12:
        return np.nan, np.nan
    theta_hat = np.mean(d_res * y_res) / jacobian
    score = d_res * (y_res - theta_hat * d_res)
    influence = score / jacobian
    se_hat = np.sqrt(np.var(influence, ddof=1) / len(d_res))
    return float(theta_hat), float(se_hat)


def fit_median_dml(d_res: np.ndarray, y_res: np.ndarray):
    """第二阶段 LAD 斜率，并用零点核密度估计构造 sandwich 标准误。"""
    keep = np.abs(d_res) > 1e-10
    if not np.any(keep):
        return np.nan, np.nan

    ratios = y_res[keep] / d_res[keep]
    weights = np.abs(d_res[keep])
    theta_hat = weighted_median(ratios, weights)
    residual = y_res - theta_hat * d_res
    n = len(residual)

    # MAD 比普通标准差更不易被重尾极值放大，用它选择核带宽
    mad = np.median(np.abs(residual - np.median(residual))) / 0.67448975
    robust_scale = max(float(mad), 1e-6)
    bandwidth = max(1.06 * robust_scale * n ** (-1.0 / 5.0), 1e-6)

    kernel_at_zero = norm.pdf(residual / bandwidth) / bandwidth
    A_hat = np.mean((d_res**2) * kernel_at_zero)
    B_hat = 0.25 * np.mean(d_res**2)
    se_hat = np.sqrt(B_hat / (A_hat**2 * n)) if A_hat > 1e-12 else np.nan
    return float(theta_hat), float(se_hat)


def estimate_both_dml(X, D, Y, seed: int):
    """在同一份样本与相同折数设定下估计两类 DML。"""
    estimates = []

    d_mean, y_mean = cross_fitted_residuals(X, D, Y, "mean", N_FOLDS, seed)
    theta_hat, se_hat = fit_mean_dml(d_mean, y_mean)
    estimates.append(("Mean_DML", theta_hat, se_hat))

    d_median, y_median = cross_fitted_residuals(X, D, Y, "median", N_FOLDS, seed)
    theta_hat, se_hat = fit_median_dml(d_median, y_median)
    estimates.append(("Median_DML", theta_hat, se_hat))
    return estimates

## 3. Monte Carlo 主循环

每个 `(DGP, error_distribution, replication)` 使用确定且互不重叠的随机种子。两种估计量共享同一份生成样本，形成配对比较。

In [ ]:
def one_replication(dgp_name: str, error_case, replication: int, n_use: int):
    """完成一个场景的一次重复，返回两个估计量的记录。"""
    dgp_id = DGP_NAMES.index(dgp_name)
    error_id = ERROR_GRID.index(error_case)
    seed = BASE_SEED + 100_000 * dgp_id + 1_000 * error_id + replication

    X, D, Y, U = generate_data(n_use, error_case, dgp_name, seed)
    estimates = estimate_both_dml(X, D, Y, seed=seed + 500_000)

    rows = []
    for estimator, theta_hat, se_hat in estimates:
        ci_low = theta_hat - 1.96 * se_hat
        ci_high = theta_hat + 1.96 * se_hat
        rows.append(
            {
                "dgp": dgp_name,
                "p": P,
                "n": n_use,
                "error_distribution": (
                    NORMAL_BASELINE if error_case == NORMAL_BASELINE else f"t({error_case:g})"
                ),
                "tail_order": error_id,
                "t_df": np.nan if error_case == NORMAL_BASELINE else float(error_case),
                "replication": replication,
                "estimator": estimator,
                "theta_hat": theta_hat,
                "se_hat": se_hat,
                "ci_low": ci_low,
                "ci_high": ci_high,
                "covered": bool(ci_low <= THETA_0 <= ci_high),
                "sample_u_variance": np.var(U, ddof=1),
                "sample_max_abs_u": np.max(np.abs(U)),
            }
        )
    return rows


def run_simulation():
    """运行 Student-t 网格与正态基准；快速模式只缩小样本量和重复次数。"""
    n_use = QUICK_N if QUICK_TEST else N
    r_use = QUICK_REPLICATIONS if QUICK_TEST else N_REPLICATIONS
    tasks = [
        (dgp_name, error_case, replication, n_use)
        for dgp_name in DGP_NAMES
        for error_case in ERROR_GRID
        for replication in range(r_use)
    ]

    mode = "QUICK_TEST" if QUICK_TEST else "FORMAL"
    print(
        f"开始运行 [{mode}]：p={P}, n={n_use}, DGP={len(DGP_NAMES)}, "
        f"error scenarios={ERROR_GRID}, repetitions={r_use}"
    )
    start = time.perf_counter()
    nested_rows = Parallel(n_jobs=N_JOBS, verbose=5)(
        delayed(one_replication)(*task) for task in tasks
    )
    rows = [row for pair in nested_rows for row in pair]
    elapsed = time.perf_counter() - start
    print(f"完成，用时 {elapsed:.1f} 秒，共 {len(rows)} 条估计结果。")
    return pd.DataFrame(rows)


results = run_simulation()
results.head()

## 4. Bias、SD、RMSE、平均 SE 与 95% coverage

正式解释 coverage 前，应同时比较 Monte Carlo `sd` 与平均报告标准误 `mean_se`。快速模式只用于验证代码，不适合据此下统计结论。

In [ ]:
def summarize_simulation(results_df: pd.DataFrame) -> pd.DataFrame:
    """按 DGP、误差分布和估计量汇总 Monte Carlo 指标。"""
    work = results_df.copy()
    work["error"] = work["theta_hat"] - THETA_0
    work["squared_error"] = work["error"] ** 2

    summary = (
        work.groupby(
            ["dgp", "error_distribution", "tail_order", "estimator"],
            as_index=False,
        )
        .agg(
            t_df=("t_df", "first"),
            mean_estimate=("theta_hat", "mean"),
            bias=("error", "mean"),
            sd=("theta_hat", "std"),
            mean_se=("se_hat", "mean"),
            mse=("squared_error", "mean"),
            coverage=("covered", "mean"),
            mean_sample_u_variance=("sample_u_variance", "mean"),
            mean_sample_max_abs_u=("sample_max_abs_u", "mean"),
            valid_runs=("theta_hat", "count"),
        )
    )
    summary["rmse"] = np.sqrt(summary.pop("mse"))
    return summary


summary = summarize_simulation(results)
summary_display = summary.sort_values(["dgp", "tail_order", "estimator"]).round(
    {
        "mean_estimate": 4,
        "bias": 4,
        "sd": 4,
        "mean_se": 4,
        "rmse": 4,
        "coverage": 3,
        "mean_sample_u_variance": 3,
        "mean_sample_max_abs_u": 2,
    }
)
display(summary_display)

In [ ]:
# 横轴从极重尾 Student-t 逐渐过渡到 N(0,1) baseline
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
plot_specs = [("bias", "Bias"), ("rmse", "RMSE"), ("coverage", "95% CI coverage")]

for ax, (metric, label) in zip(axes, plot_specs):
    sns.lineplot(
        data=summary, x="tail_order", y=metric, hue="estimator", style="dgp",
        markers=True, dashes=True, ax=ax,
    )
    ax.set_xticks(range(len(ERROR_GRID)))
    ax.set_xticklabels(
        [f"t({x:g})" if x != NORMAL_BASELINE else NORMAL_BASELINE for x in ERROR_GRID],
        rotation=30,
    )
    ax.set_xlabel("Error distribution (left = heavier tail)")
    ax.set_ylabel(label)
    if metric == "bias":
        ax.axhline(0.0, color="black", linewidth=1, linestyle=":")
    if metric == "coverage":
        ax.axhline(0.95, color="black", linewidth=1, linestyle=":")
        ax.set_ylim(0.0, 1.02)

plt.tight_layout()
plt.show()

In [ ]:
# 可选保存；SAVE_RESULTS=False 时不会生成任何 CSV
if SAVE_RESULTS:
    results.to_csv(RESULTS_FILE, index=False)
    summary.to_csv(SUMMARY_FILE, index=False)
    print(f"已保存：{Path(RESULTS_FILE).resolve()}")
    print(f"已保存：{Path(SUMMARY_FILE).resolve()}")
else:
    print("SAVE_RESULTS=False：未生成额外文件。")

## 正式运行前检查

1. 先保持 `QUICK_TEST=True`，从头运行并确认全部场景无报错。
2. 正式实验时改为 `QUICK_TEST=False`；当前使用 `N=1000` 与 `N_REPLICATIONS=1000`。
3. 不要根据快速模式的 2 次重复解释 Bias、SD、RMSE 或 coverage。
4. `df<=4` 的 Student-t 四阶矩不存在，尤其 `df` 接近 2 时 Monte Carlo 指标可能剧烈波动。
5. `N(0,1)` 是单位方差 baseline，用于判断重尾本身造成的性能变化。
6. 后续做 lognormal 时建议另建 notebook；lognormal 本身右偏且均值不为 0，需要明确中心化方式，不要直接混入本实验。